# Examples for **Multivariable Burchnall-Chaundy theory**

This is a paper written by Emma Previato.

In [ ]:
import sys
sys.path.insert(0, "../..") # dalgebra is here

from dalgebra import *
from dalgebra.commutators.spectral import spectral_operators
from dalgebra.commutators.partial import *

R = DifferentialRing(QQ['x', 'y'], [1,0], [0,1])
C = DifferentialRing(QQ)
DO.<f,q> = DifferentialPolynomialRing(R.fraction_field())
dx = lambda p : p.derivative(0)
dy = lambda p : p.derivative(1)
x,y = R.gens()

HCP = CommutingPDOHighest([f[(1,0)],f[(0,1)]], f, C.to_sage())

## TODO (partial): make sure this notebook is complete and correct after the implementation of the module dalgebra.commutators.partial

%display latex

#### Schrödinger operator with one potential

Let us think a bit about the commutation conditions for Schrödinger operator in two variables with one potential value:
$$L = -\Delta + q(x,y) = -\partial_x^2 - \partial_y^2 + q(x,y).$$

In [3]:
L = -f[(2,0)] - f[(0,2)] + q[0]
L

-f_0_2 - f_2_0 + q_0_0

IOStream.flush timed out
IOStream.flush timed out


###### Checking order 1

Let us see when there is an element that commutes with $L$ of order $1$:

In [13]:
mons = HCP.P(0) + HCP.P(1)
DOE = DO.append_variables(*[f"p_{i}" for i in range(len(mons))])
fe,p,qe = DOE.gen("f"), DOE.gens()[1:-1], DOE.gen("q")
LE = DOE(L)
A = sum(pp[0]*m for (pp,m) in zip(p,mons))
A

f_0_0*p_0_0_0 + f_0_1*p_2_0_0 + f_1_0*p_1_0_0

$$A = p_1(x,y) \partial_x + p_2(x,y)\partial_y + p_0(x,y)$$

In [14]:
LB = LE.lie_bracket(A,fe)
LB

-f_0_0*p_0_0_2 - f_0_0*p_0_2_0 - 2*f_0_1*p_0_0_1 - f_0_1*p_2_0_2 - f_0_1*p_2_2_0 - 2*f_0_2*p_2_0_1 - 2*f_1_0*p_0_1_0 - f_1_0*p_1_0_2 - f_1_0*p_1_2_0 - 2*f_1_1*p_1_0_1 - 2*f_1_1*p_2_1_0 - 2*f_2_0*p_1_1_0 - p_0_0_0*q_0_0 - p_1_0_0*q_1_0 - p_2_0_0*q_0_1 + q_0_0

In [15]:
equations = LB.coefficients(fe)
for el in list(zip(LB.monomials(fe),LB.coefficients(fe))):
    show(el)

(1, -p_0_0_0*q_0_0 - p_1_0_0*q_1_0 - p_2_0_0*q_0_1 + q_0_0)

(f_0_0, -p_0_0_2 - p_0_2_0)

(f_0_1, -2*p_0_0_1 - p_2_0_2 - p_2_2_0)

(f_0_2, -2*p_2_0_1)

(f_1_0, -2*p_0_1_0 - p_1_0_2 - p_1_2_0)

(f_1_1, -2*p_1_0_1 - 2*p_2_1_0)

(f_2_0, -2*p_1_1_0)

Let us analyze this system:
* Last equation says that $\partial_x(p_1) = 0$, so $p_1$ only depends on $y$. $\longrightarrow p_1 = p_1(y)$
* Equation [3] also provides $\partial_y(p_2) = 0$, so $p_2$ only depends on $x$. $\longrightarrow p_2 = p_2(x)$
* Equation [5] now gives a link between these two coefficients. Namely: $\partial_y(p_1) = -\partial_x(p_2)$. Since $p_1$ only depends on $y$, $\partial_y(p_1)$ is again a function in $y$. Similarly, $\partial_x(p_2)$ is a function of $x$. Hence, this identity tells us that $p_1$ is a polynomial of degree $1$ in $y$ and the same (with the variable $x$ will happen to $p_2$. $\longrightarrow p_1(y) = a + by,\quad p_2(x) = c+dx$.
* Using all previous results, it is clear that $\partial_y^2(p_1) = 0$ and $\partial_x^2(p_2) = 0$. Hence, equations [2] and [4] lead to $\partial_x(p_0) = \partial_y(p_0) = 0$. Hence, $p_0$ is a constant, making equation [1] irrelevant. $\longrightarrow p_0 = e$ 
* Finally, equation [0] lead to the equation that $q(x,y)$ must satisfy in order the have a centralizer of order 1:
  $$(1-e)q(x,y) - (a+by)\partial_x(q_0)(x,y) - (c+dx)\partial_y(q_0)(x,y) = 0,$$
  for some constants $a,b,c,d,e$. (Check https://www.wolframalpha.com/input?i=%281-a_1%29*q%28x%2Cy%29+-+%28a_2%2Ba_3*y%29*d%28q%28x%2Cy%29%29%2Fdx+-+%28a_4+%2B+a_5*x%29*d%28q%28x%2Cy%29%29%2Fdy+%3D%3D+0&lang=es)

###### Checking order 3

To avoid a big operator with lots of possible coefficients, we are going to start by taking an homogeneous order 3 operator and see what conditions are required to annihilate the highest order part of the commutator:

In [57]:
mons = HCP.P(0) + HCP.P(3)
DOE = DO.append_variables(*(["p_0"] + [f"p_{i+3}" for i in range(len(mons)-1)]))
fe,p_0,p,qe = DOE.gen("f"), DOE.gen("p_0"), DOE.gens()[2:-1], DOE.gen("q")
LE = DOE(L)
A = sum(pp[0]*m for (pp,m) in zip(tuple([p_0])+p,mons))
A

f_0_0*p_0_0_0 + f_0_3*p_6_0_0 + f_1_2*p_5_0_0 + f_2_1*p_4_0_0 + f_3_0*p_3_0_0

In [58]:
LB = LE.lie_bracket(A,fe)
LB

-f_0_0*p_0_0_2 - f_0_0*p_0_2_0 - 2*f_0_1*p_0_0_1 - f_0_3*p_6_0_2 - f_0_3*p_6_2_0 - 2*f_0_4*p_6_0_1 - 2*f_1_0*p_0_1_0 - f_1_2*p_5_0_2 - f_1_2*p_5_2_0 - 2*f_1_3*p_5_0_1 - 2*f_1_3*p_6_1_0 - f_2_1*p_4_0_2 - f_2_1*p_4_2_0 - 2*f_2_2*p_4_0_1 - 2*f_2_2*p_5_1_0 - f_3_0*p_3_0_2 - f_3_0*p_3_2_0 - 2*f_3_1*p_3_0_1 - 2*f_3_1*p_4_1_0 - 2*f_4_0*p_3_1_0 - p_0_0_0*q_0_0 - p_3_0_0*q_3_0 - p_4_0_0*q_2_1 - p_5_0_0*q_1_2 - p_6_0_0*q_0_3 + q_0_0

In [59]:
equations = [c for (m,c) in zip(LB.monomials(fe),LB.coefficients(fe)) if DOE(m).order(fe) == LB.order(fe)]
for el in equations:
    show(el)

-2*p_6_0_1

-2*p_5_0_1 - 2*p_6_1_0

-2*p_4_0_1 - 2*p_5_1_0

-2*p_3_0_1 - 2*p_4_1_0

-2*p_3_1_0

This system is similar to the case of order 3, but slightly more intricate:
* $p_3(x,y) = p_3(y)$ (equation [4])
* $p_6(x,y) = p_6(x)$ (equation [0])
* $\partial_x(p_6) = -\partial_y(p_5)$, hence $p_5(x,y) = a(x) - \partial_x(p_6(x))y$. (equation [1])
* Similarly, $p_4(x,y) = b(y) - \partial_y(p_3(y))x$. (equation [3]).
* Equation [2] provides $\partial_y(p_4(x,y)) = - \partial_x(p_5(x,y))$. This leads to:
  

In [90]:
DOEC = DOE.add_constants("a_0","a_1","a_2","b_0","b_1","b_2","c_0","c_1","d_0","d_1","e")
xc,yc,a,b,c,d,e = DOEC.base().gens()[0], DOEC.base().gens()[1], DOEC.base().gens()[2:5], DOEC.base().gens()[5:8], DOEC.base().gens()[8:10], DOEC.base().gens()[10:12], DOEC.base().gens()[12]
fec = DOEC.gen("f")

In [91]:
p6 = c[0]+xc*c[1]
p3 = d[0]+yc*d[1]
ax = a[0] + a[1]*xc 
by = b[0] - a[1]*yc

In [92]:
for el in [equ(p_3=p3, p_4=by - xc*dy(p3), p_5=ax - yc*dx(p6), p_6=p6) for equ in equations]:
    show(el)

0

0

0

0

0

In [93]:
LB_red1 = DOEC(LB)(p_3=p3, p_4=by - xc*dy(p3), p_5=ax - yc*dx(p6), p_6=p6, p_0=e)
P = LB_red1.parent(); fec1 = P.gen("f")
equations_1 = [c for (m,c) in zip(LB_red1.monomials(fec1), LB_red1.coefficients(fec1)) if P(m).order(fec1) == LB_red1.order(fec1)]

In [95]:
equations_1 # this is the condition on the potential to have a non-trivial centralizer of order 3

[-(e - 1)*q_0_0 - (x*c_1 + c_0)*q_0_3 - (x*a_1 - y*c_1 + a_0)*q_1_2 + (y*a_1 + x*d_1 - b_0)*q_2_1 - (y*d_1 + d_0)*q_3_0]